In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import recall_score, precision_score, f1_score, classification_report, roc_auc_score
import joblib
import os


df = pd.read_csv('../output/preprocessed_data.csv')
df.head()

,amount_src,amount_usd,fee,exchange_rate_src_to_dest,ip_risk_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,...,channel_WEB,ip_country_CA,ip_country_UK,ip_country_UNKNOWN,ip_country_US,new_device,location_mismatch,hour_sin,hour_cos,is_fraud
0,0.558773,0.543715,0.274827,-0.880988,-1.339161,-0.207593,-0.157747,-0.357508,-0.491316,-0.584829,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-1.000000,-1.836970e-16,0
1,0.283074,-0.038214,0.271402,0.234591,0.948103,-0.207593,0.224151,-0.357508,1.874437,-0.584829,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.866025,5.000000e-01,0
2,0.013657,0.002302,-0.441794,-0.048752,0.494635,-0.207593,-0.624739,-0.357508,-0.491316,-0.584829,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.258819,9.659258e-01,0
3,-1.179389,-1.174986,-0.781820,-1.039431,1.021055,-0.207593,1.023780,-0.357508,-0.491316,-0.584829,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.258819,9.659258e-01,0
4,0.246329,0.233093,0.029904,0.979515,-1.356230,-0.207593,0.136044,-0.357508,-0.491316,-0.584829,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.707107,-7.071068e-01,0


### Data Splitting

In [3]:
# Splitting target and features
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

# Splitting train and test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

smote = SMOTE(random_state=42)

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

pd.Series(y_train_res).value_counts()

is_fraud
1    7734
0    7734
Name: count, dtype: int64

### Random Forest Classifier

In [4]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42)

model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]
print('\n')
print("Recall:", recall_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print('\n')
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))



Recall: 0.8367346938775511
Precision: 0.9704142011834319
F1-Score: 0.8986301369863015


              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1934
           1       0.97      0.84      0.90       196

    accuracy                           0.98      2130
   macro avg       0.98      0.92      0.94      2130
weighted avg       0.98      0.98      0.98      2130

ROC-AUC: 0.9523022497520208


### Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(random_state=42)
model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print('\n')
print("Recall:", recall_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print('\n')
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))



Recall: 0.8673469387755102
Precision: 0.6938775510204082
F1-Score: 0.7709750566893425



Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.97      1934
           1       0.69      0.87      0.77       196

    accuracy                           0.95      2130
   macro avg       0.84      0.91      0.87      2130
weighted avg       0.96      0.95      0.95      2130

ROC-AUC: 0.9598827638604562


C:\Users\godfr\anaconda3\envs\ml_env\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
# !pip install xgboost
# !pip install lightgbm

In [7]:
import xgboost as xgb

# Convert to DMatrix (optimized data structure)
dtrain = xgb.DMatrix(X_train_res, label=y_train_res)
dtest = xgb.DMatrix(X_test, label=y_test)

# Set parameters
params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "eta": 0.1,
    "max_depth": 4
}

# Train model
model_xgb = xgb.train(params, dtrain, num_boost_round=100)

# Predict
y_pred_prob = model_xgb.predict(dtest)
y_pred = (y_pred_prob > 0.5).astype(int)

# Evaluate
print('\n')
print("Recall:", recall_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print('\n')
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_proba))



Recall: 0.8367346938775511
Precision: 0.9590643274853801
F1-Score: 0.8937329700272479



Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1934
           1       0.96      0.84      0.89       196

    accuracy                           0.98      2130
   macro avg       0.97      0.92      0.94      2130
weighted avg       0.98      0.98      0.98      2130

ROC-AUC: 0.9598827638604562


In [9]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

# LightGBM parameters
params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 32,
    "max_depth": 6,
    "is_unbalance": True,  # same as class_weight="balanced"
    "verbose": -1
}

model_lgb = LGBMClassifier(
    **params,
    n_estimators=300
)

# X_train, y_train are numpy/pandas arrays
# Fit the model
model_lgb.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        early_stopping(stopping_rounds=50),
        log_evaluation(50)  # logs every 50 rounds
    ]
)

# Predictions
y_pred_prob = model_lgb.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# Evaluation
print('\n')
print("Recall:", recall_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("F1-Score:", f1_score(y_test, y_pred))
print('\n')
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_proba))


# Save model
os.makedirs('../artifacts/model', exist_ok=True)
joblib.dump(model_lgb, "../artifacts/model/fraud_model.pkl")

Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.952414
Early stopping, best iteration is:
[40]	valid_0's auc: 0.953886


Recall: 0.8418367346938775
Precision: 0.9217877094972067
F1-Score: 0.88



Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1934
           1       0.92      0.84      0.88       196

    accuracy                           0.98      2130
   macro avg       0.95      0.92      0.93      2130
weighted avg       0.98      0.98      0.98      2130

ROC-AUC: 0.9598827638604562


['../artifacts/model/fraud_model.pkl']